In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
from urllib.parse import urlparse
import re
from tqdm import tqdm

In [22]:
# Load your existing dataset
df = pd.read_csv('data/processed/ted_talks_all_clean.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()
df_test = df.head(10)  # Test with first 10 rows

Dataset shape: (7244, 4)
Columns: ['title', 'speaker', 'duration', 'url']


In [ ]:
def scrape_single_ted_talk(url, headers):
    """
    Scrape detailed information from a single TED talk page
    """
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the __NEXT_DATA__ script
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        
        # Navigate to talk data (structure may vary)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            # Try alternative path
            talk_data = data.get('props', {}).get('pageProps', {})
        
        # Extract relevant fields
        result = {
            'id': talk_data.get('id'),
            'title': talk_data.get('title'),
            'speaker': talk_data.get('presenterDisplayName'),
            'description': talk_data.get('description'),
            'duration': talk_data.get('duration'),
            'duration_min': talk_data.get('duration', 0) // 60,  # Convert to minutes
            'video_context': talk_data.get('videoContext'),
            'type': talk_data.get('type', {}).get('name'),
            'url': talk_data.get('canonicalUrl'),
            'published_at': talk_data.get('publishedAt'),
            'views': talk_data.get('viewedCount'),
            'slug': talk_data.get('slug'),
        }
        
        # Extract topics
        topics = talk_data.get('topics', {})
        if isinstance(topics, dict):
            topic_nodes = topics.get('nodes', [])
        else:
            topic_nodes = topics if isinstance(topics, list) else []
            
        topic_names = [t.get('name') for t in topic_nodes if isinstance(t, dict)]
        result['topics'] = ', '.join(topic_names)
        result['num_topics'] = len(topic_names)
        
        # Extract image
        images = talk_data.get('primaryImageSet', [])
        if images:
            for img in images:
                if img.get('aspectRatioName') == '16x9':
                    result['image_url'] = img.get('url')
                    break
        
        # Viewing percentages
        result['tedcom_percentage'] = talk_data.get('tedcomPercentage')
        result['youtube_percentage'] = talk_data.get('youtubePercentage')
        result['podcasts_percentage'] = talk_data.get('podcastsPercentage')
        
        return result
        
    except Exception as e:
        print(f"Error scraping {url}: {str(e)}")
        return None

In [24]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

results = []
for idx, row in df_test.iterrows():
    print(f"{idx+1}/{len(df_test)}: {row['title'][:50]}...", end=" ")
    
    extra_data = scrape_single_ted_talk(row['url'], headers)
    
    combined = {
        'title': row['title'],
        'speaker': row['speaker'],
        'duration': row['duration'],
        'url': row['url'],
    }
    if extra_data:
        combined.update(extra_data)
        print("✓")
    else:
        print("✗")
    
    results.append(combined)
    time.sleep(1)

df_result = pd.DataFrame(results)
print(f"\nScraped {df_result['id'].notna().sum()}/{len(df_result)} successfully")
df_result

1/10: A pastry chef works his chocolatier magic \'97 liv... ✓
2/10: The flourishing future of women's sports... ✓
3/10: How we\'92re turning pollution into toys, toothpas... ✓
4/10: The best thing that could happen to the energy ind... ✓
5/10: 3 simple ways to build stronger relationships at w... ✓
6/10: How video games can power up your parenting... ✓
7/10: Why we need to know our lives matter... ✓
8/10: How nearly dying helped me discover my own cure (a... ✓
9/10: Could we detect breast cancer with a fingerprint?... ✓
10/10: Why you should spend less time with your kids... ✓

Scraped 10/10 successfully


,title,speaker,duration,url,id,description,duration_min,video_context,type,published_at,views,slug,topics,num_topics,image_url,tedcom_percentage,youtube_percentage,podcasts_percentage
0,A pastry chef works his chocolatier magic — live,Amaury Guichon,757,https://www.ted.com/talks/amaury_guichon_a_pas...,155034,Get a taste of the chocolatier life from world...,12,TED2025,TED Stage Talk,2025-10-14T14:59:25Z,103857,amaury_guichon_a_pastry_chef_works_his_chocola...,"design, innovation, food, creativity, art",5,https://talkstar-assets.s3.amazonaws.com/produ...,0.004,0.0,0.993
1,The flourishing future of women's sports,Kate Johnson,772,https://www.ted.com/talks/kate_johnson_the_flo...,162374,Women's sports are surging in popularity aroun...,12,TEDSports Indianapolis 2025,TED Stage Talk,2025-10-13T15:09:12Z,169777,kate_johnson_the_flourishing_future_of_women_s...,"culture, technology, media, sports, AI, algorithm",6,https://talkstar-assets.s3.amazonaws.com/produ...,0.014,0.0,0.98
2,"How we’re turning pollution into toys, toothpa...",Xu Hao,781,https://www.ted.com/talks/xu_hao_how_we_re_tur...,160684,It took alcohol 200 years to go from scientifi...,13,TED Countdown Summit 2025,TED Stage Talk,2025-10-09T14:45:26Z,220499,xu_hao_how_we_re_turning_pollution_into_toys_t...,"climate change, science, sustainability, techn...",8,https://talkstar-assets.s3.amazonaws.com/produ...,0.025,0.0,0.961
3,The best thing that could happen to the energy...,Matt Tilleard,769,https://www.ted.com/talks/matt_tilleard_the_be...,161395,History has been written by whoever controls t...,12,TED Countdown Summit 2025,TED Stage Talk,2025-10-02T14:53:50Z,239945,matt_tilleard_the_best_thing_that_could_happen...,"climate change, politics, sustainability, ener...",9,https://talkstar-assets.s3.amazonaws.com/produ...,0.04,0.0,0.931
4,3 simple ways to build stronger relationships ...,Alyssa Birnbaum,903,https://www.ted.com/talks/alyssa_birnbaum_3_si...,161270,Doing the best at your job isn't just about wo...,15,TEDxClaremontGraduateUniversity,TEDx Talk,2025-09-23T14:51:23Z,293018,alyssa_birnbaum_3_simple_ways_to_build_stronge...,"business, psychology, relationships, communica...",6,https://talkstar-assets.s3.amazonaws.com/produ...,0.13,0.0,0.829
5,How video games can power up your parenting,Hannah Boquet,856,https://www.ted.com/talks/hannah_boquet_how_vi...,160410,Parenting an eye-rolling teenager glued to a g...,14,TEDxSioux Falls,TED Stage Talk,2025-09-17T14:49:12Z,258930,hannah_boquet_how_video_games_can_power_up_you...,"technology, entertainment, relationships, pare...",9,https://talkstar-assets.s3.amazonaws.com/produ...,0.075,0.0,0.895
6,Why we need to know our lives matter,Jennifer Wallace,751,https://www.ted.com/talks/jennifer_wallace_why...,148433,It’s not enough to do important work — we need...,12,TED2025,TED Stage Talk,2025-09-10T14:49:18Z,314548,jennifer_wallace_why_we_need_to_know_our_lives...,"culture, community, work, personal growth, soc...",6,https://talkstar-assets.s3.amazonaws.com/produ...,0.062,0.099,0.815
7,How nearly dying helped me discover my own cur...,David Fajgenbaum,840,https://www.ted.com/talks/david_fajgenbaum_how...,157718,Physician-scientist David Fajgenbaum was dying...,14,TED2025,TED Stage Talk,2025-09-08T20:13:52Z,398079,david_fajgenbaum_how_nearly_dying_helped_me_di...,"science, technology, disease, health, health c...",8,https://talkstar-assets.s3.amazonaws.com/produ...,0.172,0.195,0.609
8,Could we detect breast cancer with a fingerprint?,Simona Francese,749,https://www.ted.com/talks/simona_francese_coul...,159724,Breast cancer is the most common cancer among ...,12,TEDxManchester,TEDx Talk,2025-08-27T14:17:06Z,261307,simona_francese_could_we_detect_breast_cancer_...,"science, technology, health, health care, canc...",9,https://talkstar-assets.s3.amazonaws.com/produ...,0.066,0.0,0.914
9,Why you should spend less time with your kids,Lenore Skenazy,794,https://www.ted.com/talks/lenore_skenazy_why_y...,153162,"Whether it’s micromanaging p

In [32]:
def scrape_single_ted_talk(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})

        print(f"\nAvailable keys in talk_data:")
        print(list(talk_data.keys()))
    
    
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        return None

In [33]:
scrape_single_ted_talk('https://www.ted.com/talks/sir_ken_robinson_do_schools_kill_creativity', headers)


Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations', 'featured', 'customPartnerContent', 'topics', 'presenterDisplayName', 'duration', 'canonicalUrl', 'viewedCount', 'tedcomPercentage', 'youtubePercentage', 'podcastsPercentage', 'tedappsPercentage', 'publishedAt', 'id', 'title', 'slug', 'primaryImageSet']


In [28]:
def scrape_single_ted_talk(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})

        print(f"\nAvailable keys in talk_data:")
        print(list(talk_data.keys())[:20])
        
        # Extract topics
        topics = talk_data.get('topics', {})
        topic_nodes = topics.get('nodes', []) if isinstance(topics, dict) else topics
        topic_names = [t.get('name') for t in topic_nodes if isinstance(t, dict)]
        
        # Extract ratings (Inspiring, Informative, etc.)
        ratings = talk_data.get('ratings', [])
        rating_dict = {r.get('name'): r.get('count') for r in ratings if isinstance(r, dict)}

        return {
            'id': talk_data.get('id'),
            'title': talk_data.get('title'),
            'speaker': talk_data.get('presenterDisplayName'),
            'description': talk_data.get('description'),
            'recorded_at': talk_data.get('recordedAt'),  # Event date
            'duration': talk_data.get('duration'),
            'duration_min': talk_data.get('duration', 0) // 60,  # Convert to minutes
            'views': talk_data.get('viewedCount'),
            'topics': ', '.join(topic_names),
            'num_topics': len(topic_names),
            'video_context': talk_data.get('videoContext'),  # Event name
            'type': talk_data.get('type', {}).get('name') if isinstance(talk_data.get('type'), dict) else None,
            'language': talk_data.get('language'),
            'num_subtitles': len(talk_data.get('translations', [])),  # Available languages
            'url': talk_data.get('canonicalUrl'),
            
            # Engagement metrics
            'tedcom_percentage': talk_data.get('tedcomPercentage'),
            'youtube_percentage': talk_data.get('youtubePercentage'),
            'podcasts_percentage': talk_data.get('podcastsPercentage'),
        }
    
    
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        return None

In [29]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

results = []
for idx, row in df_test.iterrows():
    print(f"{idx+1}/{len(df_test)}: {row['title'][:50]}...", end=" ")
    
    extra_data = scrape_single_ted_talk(row['url'], headers)
    
    combined = {
        'title': row['title'],
        'speaker': row['speaker'],
        'duration': row['duration'],
        'url': row['url'],
    }
    if extra_data:
        combined.update(extra_data)
        print("✓")
    else:
        print("✗")
    
    results.append(combined)
    time.sleep(1)

df_result = pd.DataFrame(results)
print(f"\nScraped {df_result['id'].notna().sum()}/{len(df_result)} successfully")
df_result

1/10: A pastry chef works his chocolatier magic \'97 liv... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations']
✓
2/10: The flourishing future of women's sports... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations']
✓
3/10: How we\'92re turning pollution into toys, toothpas... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVi

,title,speaker,duration,url,id,description,recorded_at,duration_min,views,topics,num_topics,video_context,type,language,num_subtitles,tedcom_percentage,youtube_percentage,podcasts_percentage
0,A pastry chef works his chocolatier magic — live,Amaury Guichon,757,https://www.ted.com/talks/amaury_guichon_a_pas...,155034,Get a taste of the chocolatier life from world...,None,12,103857,"design, innovation, food, creativity, art",5,TED2025,TED Stage Talk,en,0,0.004,0.0,0.993
1,The flourishing future of women's sports,Kate Johnson,772,https://www.ted.com/talks/kate_johnson_the_flo...,162374,Women's sports are surging in popularity aroun...,None,12,169777,"culture, technology, media, sports, AI, algorithm",6,TEDSports Indianapolis 2025,TED Stage Talk,en,0,0.014,0.0,0.98
2,"How we’re turning pollution into toys, toothpa...",Xu Hao,781,https://www.ted.com/talks/xu_hao_how_we_re_tur...,160684,It took alcohol 200 years to go from scientifi...,None,13,220499,"climate change, science, sustainability, techn...",8,TED Countdown Summit 2025,TED Stage Talk,en,0,0.025,0.0,0.961
3,The best thing that could happen to the energy...,Matt Tilleard,769,https://www.ted.com/talks/matt_tilleard_the_be...,161395,History has been written by whoever controls t...,None,12,239945,"climate change, politics, sustainability, ener...",9,TED Countdown Summit 2025,TED Stage Talk,en,0,0.04,0.0,0.931
4,3 simple ways to build stronger relationships ...,Alyssa Birnbaum,903,https://www.ted.com/talks/alyssa_birnbaum_3_si...,161270,Doing the best at your job isn't just about wo...,None,15,293018,"business, psychology, relationships, communica...",6,TEDxClaremontGraduateUniversity,TEDx Talk,en,0,0.13,0.0,0.829
5,How video games can power up your parenting,Hannah Boquet,856,https://www.ted.com/talks/hannah_boquet_how_vi...,160410,Parenting an eye-rolling teenager glued to a g...,None,14,258930,"technology, entertainment, relationships, pare...",9,TEDxSioux Falls,TED Stage Talk,en,0,0.075,0.0,0.895
6,Why we need to know our lives matter,Jennifer Wallace,751,https://www.ted.com/talks/jennifer_wallace_why...,148433,It’s not enough to do important work — we need...,None,12,314548,"culture, community, work, personal growth, soc...",6,TED2025,TED Stage Talk,en,0,0.062,0.099,0.815
7,How nearly dying helped me discover my own cur...,David Fajgenbaum,840,https://www.ted.com/talks/david_fajgenbaum_how...,157718,Physician-scientist David Fajgenbaum was dying...,None,14,398079,"science, technology, disease, health, health c...",8,TED2025,TED Stage Talk,en,0,0.172,0.195,0.609
8,Could we detect breast cancer with a fingerprint?,Simona Francese,749,https://www.ted.com/talks/simona_francese_coul...,159724,Breast cancer is the most common cancer among ...,None,12,261307,"science, technology, health, health care, canc...",9,TEDxManchester,TEDx Talk,en,0,0.066,0.0,0.914
9,Why you should spend less time with your kids,Lenore Skenazy,794,https://www.ted.com/talks/lenore_skenazy_why_y...,153162,"Whether it’s micromanaging playtime, constantl...",None,13,458296,"education, social change, parenting, personal ...",6,TED2025,TED Stage Talk,en,0,0.146,0.227,0.584
